# Step-by-step demo of NEDAS using the vort2d model


## Install NEDAS and dependencies
NEDAS is available from the PyPI platform. You can install with ```pip install NEDAS```.

You can also clone the `NEDAS` repository and install in editable mode ```cd NEDAS; pip install -e .``` for active code development.

In [1]:
# 1. Install the latest version of NEDAS on develop branch
%cd ~
%rm -rf NEDAS
!git clone https://github.com/nansencenter/NEDAS.git
%cd NEDAS
%pip install .

/root
Cloning into 'NEDAS'...
remote: Enumerating objects: 9556, done.
remote: Counting objects: 100% (774/774), done.
remote: Compressing objects: 100% (366/366), done.
remote: Total 9556 (delta 453), reused 485 (delta 402), pack-reused 8782 (from 1)
Receiving objects: 100% (9556/9556), 7.57 MiB | 18.29 MiB/s, done.
Resolving deltas: 100% (6609/6609), done.
/root/NEDAS
Processing /root/NEDAS
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for NEDAS: filename=nedas-1.2.1.dev4+g3bf60717f-py3-none-any.whl size=3386188 sha256=d708d2e423ff5c067427107004842552bf5a4bb4b8160ec7d2363ccc2be83772
  Stored in directory: /tmp/pip-ephem-wheel-cache-zg0pgv2x/wheels/d1/a4/be/a44082c2c28ad510b6366e399d332c2f80015370e5d157e399
Successfully built NEDAS
  Attempting uninstall: NEDAS
    Found existing installation: NEDAS 1.2.1.dev4+g3bf60717f
    Uninstalling NEDAS-1.2.1.dev4+g3bf60717f:
      Successfu

In [2]:
# 2. Install additional dependencies
# numba provides JIT compilation of python function to speed it up during runtime
%pip install numba
# cmocean provides better colormaps for visualization
%pip install cmocean

In [3]:
# 3. If in Google Colab, need to clone the tutorial repo too
import sys
if 'google.colab' in sys.modules:
    %cd ~
    %rm -rf NEDAS_tutorials
    !git clone https://github.com/myying/NEDAS_tutorials.git
    %cd NEDAS_tutorials

/root
Cloning into 'NEDAS_tutorials'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 79 (delta 34), reused 61 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 1.18 MiB | 12.31 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/root/NEDAS_tutorials


In [4]:
# Load utility modules
# scientific computation
import numpy as np

# a few graphic utility modules
import matplotlib.pyplot as plt
import cmocean
from matplotlib import cm
from matplotlib.animation import FuncAnimation
from IPython.core.display import HTML


## Initialize configuration and main scheme

In [5]:
# load configuration YAML file
from NEDAS.config import Config
config = Config(config_file="vort2d/config.yml")

In [6]:
# initialize the main Scheme
from NEDAS import get_scheme
scheme = get_scheme(config)

In [7]:
# the model object
model = scheme.c.models['vort2d']

from NEDAS.models.vort2d import Vort2DModel
assert isinstance(model, Vort2DModel)

In [8]:
#
dataset = scheme.c.datasets['vort2d']

from NEDAS.datasets.vort2d import Vort2DObs
assert isinstance(dataset, Vort2DObs)

In [9]:
from NEDAS.grid import RegularGrid
grid = scheme.c.grid

## Prepare truth

check nc files in vort2d/truth

In [10]:
scheme.c.time = config.time_start

In [11]:
%rm -rf vort2d/work/truth

In [12]:
model.loc_sprd = 0
scheme.run_step('prepare_truth')


Running prepare_truth step:                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
├── Generate vort2d truth ─────────────────────── ✅     8.86s                                                                                                                                          

## Prepare initial ensemble and run forecasts

In [13]:
%rm -rf vort2d/work/init_ens

In [14]:
model.loc_sprd = 100000
scheme.run_step('prepare_init_ensemble')


Running prepare_init_ensemble step:                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            
├── Generate vort2d init ensemble ─────────────── ✅     0.84s (all 16 jobs done)                                                                                                                       

In [15]:
%rm -rf vort2d/work/cycle

In [16]:
scheme.c.progress.call_stack = []
scheme.c.progress.push('')
scheme.c.time = config.time_start
scheme.c.log_event("CYCLING START...")
while scheme.c.time < config.time_end:
    scheme.c.log_event(f"CURRENT CYCLE: {scheme.c.time} -> {scheme.c.next_time}")
    scheme.run_step('preprocess')
    scheme.run_step('ensemble_forecast')
    scheme.c.time = scheme.c.next_time
scheme.c.log_event("CYCLING COMPLETE.", flag='finish')


│   
◈ CYCLING START...
│   
◈ CURRENT CYCLE: 2001-01-01 00:00:00+00:00 -> 2001-01-01 03:00:00+00:00
├── Running preprocess step ───────────────────── ✅     0.66s (all 16 jobs done)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               
├── Running ensemble_forecast step ────────────── ✅     9.98s (all 16 jobs done)                  

In [17]:
scheme.c.time = config.time_start
truth_state = []
times = []
while scheme.c.time < config.time_end:
    times.append(scheme.c.time)

    truth = model.read_var(path = model.truth_dir, name='velocity', member=None, time=scheme.c.time)
    truth_state.append(truth)
    scheme.c.time = scheme.c.next_time

In [18]:
scheme.c.time = config.time_start
forecast_state = []
for n in range(10):
    path = scheme.c.fs.forecast_dir(scheme.c.time, 'vort2d')
    ens = []
    for m in range(scheme.c.nens):
        ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
    forecast_state.append(ens)
    scheme.c.time = scheme.c.next_time

In [ ]:
from NEDAS.utils.spatial_operation import gradx, grady

def to_vorticity(vel):
    u, v = vel[0], vel[1]
    zeta = gradx(v, grid.dx, grid.cyclic_dim) - grady(u, grid.dy, grid.cyclic_dim)
    return zeta

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
nt = 10 #len(truth_state)
clvs = [1e-3]
current_contour_sets = []

def update(n):
    global current_contour_sets
    for cs in current_contour_sets:
        cs.remove()
    current_contour_sets.clear()

    pc_truth = ax[0].pcolor(to_vorticity(truth_state[n]), vmin=-5e-3, vmax=5e-3, cmap='bwr')
    current_contour_sets.append(pc_truth)
    cmap = [plt.cm.jet(x) for x in np.linspace(0, 1, scheme.c.nens)]
    for m in range(scheme.c.nens):
        cs_ens = ax[0].contour(to_vorticity(forecast_state[n][m]), clvs, colors=[cmap[m][0:3]], linewidths=0.5)
        current_contour_sets.append(cs_ens)
    cs_truth = ax[0].contour(to_vorticity(truth_state[n]), clvs, colors='k', linewidths=1.5)
    current_contour_sets.append(cs_truth)
    ax[0].set_title(f"Forecast {n*config.cycle_period} h")
    return current_contour_sets

ani = FuncAnimation(fig, update, frames=range(nt), blit=False, interval=100)
plt.close()
HTML(ani.to_jshtml())

## Substeps in an analysis cycle

In [ ]:
scheme.c.time = config.time_start

## Running the entire scheme

Cycling from time_start to time_end

In [ ]:
scheme.c.time = config.time_start
scheme.c.progress.call_stack = []

scheme()


In [ ]:
scheme.c.time = config.time_start
prior_state = []
post_state = []
while scheme.c.time < config.time_end:

    path = scheme.c.fs.forecast_dir(scheme.c.prev_time, 'vort2d')
    ens = []
    for m in range(scheme.c.nens):
        ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
    prior_state.append(ens)

    ens = []
    for m in range(scheme.c.nens):
        if config.run_analysis and scheme.c.time >= config.time_analysis_start and scheme.c.time <= config.time_analysis_end:
            path = scheme.c.fs.forecast_dir(scheme.c.time, 'vort2d')
            ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
        else:
            ens.append(np.full(truth.shape, np.nan))
    post_state.append(ens)

    scheme.c.time = scheme.c.next_time

In [ ]:
from NEDAS.utils.spatial_operation import gradx, grady

def to_vorticity(vel):
    u, v = vel[0], vel[1]
    zeta = gradx(v, grid.dx, grid.cyclic_dim) - grady(u, grid.dy, grid.cyclic_dim)
    return zeta

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
nt = 10 #len(truth_state)
clvs = [1e-3]
current_contour_sets = []

def update(n):
    global current_contour_sets
    for cs in current_contour_sets:
        cs.remove()
    current_contour_sets.clear()

    pc_truth = ax[0].pcolormesh(to_vorticity(truth_state[n]), vmin=-5e-3, vmax=5e-3, cmap='bwr')
    current_contour_sets.append(pc_truth)
    cmap = [plt.cm.jet(x) for x in np.linspace(0, 1, scheme.c.nens)]
    for m in range(scheme.c.nens):
        cs_ens = ax[0].contour(to_vorticity(prior_state[n][m]), clvs, colors=[cmap[m][0:3]], linewidths=0.5)
        current_contour_sets.append(cs_ens)
    cs_truth = ax[0].contour(to_vorticity(truth_state[n]), clvs, colors='k', linewidths=1.5)
    current_contour_sets.append(cs_truth)
    ax[0].set_title(f"Forecast {n*config.cycle_period} h")
    return current_contour_sets

ani = FuncAnimation(fig, update, frames=range(nt), blit=False, interval=100)
plt.close()
HTML(ani.to_jshtml())

## A few things to try to play with
